# Accuracy / training-time / memory trade-off

Reads the result spreadsheets written by the noise-sweep notebooks and summarises, for every method, how much it
improves the accuracy of the **unadapted model** (the clean model $\theta_0$ evaluated on the noisy data, not adapted)
and how much faster than backpropagation it is.

One marker per method, averaged over the benchmarks: x = training time relative to backpropagation
(geometric mean of the per-model ratios, so smaller is faster), y = accuracy gain over the unadapted model
(arithmetic mean of the per-model percentages). The unadapted model is the 0 % line; backpropagation, the reference for
the training time, is drawn at 1×. The best methods sit top-left. The training time cannot be taken relative to the
unadapted model, which is not trained at all.

Nothing below is hardcoded to a particular run:

* Each model folder is scanned for `results_*` sub-folders and the **newest** `.xlsx` in each one is loaded (sheet `long`).
* Accuracy, training time and training memory are averaged over every row a method has (all noise levels and seeds).
  The accuracy gain is that average divided by the same average of the unadapted model, minus 1; time and memory are
  divided by the averages of backpropagation.
* Trailing run details such as `(T=1.0,10ep)` are stripped from the method labels; baseline rows that occur in several files are taken from the BOLSTER file.

Run it from the `plots/` folder (or anywhere inside the project).

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, FuncFormatter

In [ ]:
FONT_FAMILY         = "DejaVu Sans"   # e.g. "serif", "Times New Roman", "Arial", ...
BASE_FONTSIZE       = 20              # default text size (titles, legend title, ...)
AXIS_LABEL_FONTSIZE = 14              # x/y axis label size
AXIS_TICK_FONTSIZE  = 14              # x/y tick label size
plt.rcParams.update({
    "font.family":     FONT_FAMILY,
    "font.size":       BASE_FONTSIZE,
    "axes.labelsize":  AXIS_LABEL_FONTSIZE,
    "axes.titlesize":  BASE_FONTSIZE + 1,
    "xtick.labelsize": AXIS_TICK_FONTSIZE,
    "ytick.labelsize": AXIS_TICK_FONTSIZE,
    "legend.fontsize": BASE_FONTSIZE - 3,
})

## Configuration

Only presentation choices live here (which folders are shown, how labels are displayed, colours, markers, axis limits). All numbers come from the spreadsheets.

In [ ]:
# Model folder -> legend label. Leave empty ({}) to auto-discover every folder with results_* sub-folders.
MODELS = {
    "lenet5_mnist":             "LeNet5/MNIST",
    "resnet20_cifar10":         "ResNet20/CIFAR-10",
    "mobilenetv1_tinyimagenet": "MobileNet-v1/Tiny ImageNet",
}
# Short benchmark names for the highlight box
SHORT_NAMES = {
    "lenet5_mnist":             "LeNet5",
    "resnet20_cifar10":         "ResNet20",
    "mobilenetv1_tinyimagenet": "MobileNet-v1",
}

RESULTS_DIR_GLOB = "results_*"        # sub-folders of each model folder that hold the xlsx files
SHEET_NAME       = "long"             # sheet with one row per (noise_level, method[, seed])
ACC_COL          = "accuracy"         # y axis: accuracy gain over the accuracy reference
TIME_COL         = "train_time_sec"   # x axis: training time relative to the time reference
MEM_COL          = "train_mem_mb"     # optional: reported in the table (missing in some files)

# Label in the xlsx `method` column -> legend label. Trailing parentheses such as "(T=1.0,10ep)"
# are stripped before the lookup, so "LMC-SPSA(T=1.0,200ep)" matches the "LMC-SPSA" entry.
DISPLAY_NAMES = {
    "Full retrain":      "Backpropagation",
    "Init (no retrain)": "Unadapted model",
    "BOLSTER":           "Ours",
    "LMC-SPSA":          "LMC",
    "Sparse-MeZO":       "SMEZO",
}
EXCLUDE_METHODS  = {"BP-FC"}                        # xlsx labels (after stripping) never shown
PREFERRED_SOURCE = "bolster"                        # folder that supplies rows reported by several files
ACC_REFERENCE    = "Unadapted model"                # display name of the 0 % accuracy-gain reference
TIME_REFERENCE   = "Backpropagation"                # display name of the 1x training-time / memory reference
HIGHLIGHT_METHOD = "Ours"                           # its per-benchmark gains are written next to its marker

# Methods to draw (display names), or None for every method except the two references and the excluded ones.
METHODS_TO_PLOT = None            # e.g. ["Ours", "PEPITA", "SCFF"]
PLOT_ORDER      = ["PEPITA", "SCFF", "SMEZO", "LMC", "Ours"]   # drawing/legend order; others appended alphabetically

COLORS = {
    "Ours":            "tab:green",
    "Backpropagation": "tab:blue",
    "Unadapted model": "black",
    "SCFF":            "red",
    "PEPITA":          "tab:orange",
    "SMEZO":           "tab:purple",
    "LMC":             "tab:gray",
}



MARKER_SIZE    = 300         # marker area (points^2)
XLIM           = None        # (left, right) relative-time range, or None to derive from the data
SYMMETRIC_X    = True        # derived range is symmetric around 1x on the log axis (1x sits in the middle)
YLIM           = (-115, 400) # accuracy gain over the unadapted model [%] (room below 0 for the region labels)
REGION_LABEL_Y = 0.02        # height (axes fraction) of the "faster / slower than backpropagation" labels
LEGEND_POS     = (0.5, 1.0)      # lower-centre of the method legend (axes fraction): one row above the plot
LEGEND_NCOL    = 5               # legend columns
HIGHLIGHT_BOX  = (0.56, 0.505)    # lower-left corner of the highlight box (axes fraction)
ZERO_LABEL_X   = 0.895           # right end (axes fraction) of the "Unadapted model" label on the 0 % line
FIGSIZE        = (7, 4.5)

OUTPUT_FILE = Path("figures") / "intro_tradeoff.png"

## Locate and load the result spreadsheets

In [ ]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    """Walk up from `start` until a folder is found whose children contain results_* folders."""
    for candidate in (start, *start.parents):
        if any(p.is_dir() for p in candidate.glob(f"*/{RESULTS_DIR_GLOB}")):
            return candidate
    raise FileNotFoundError(f"No '*/{RESULTS_DIR_GLOB}' folders found in {start} or its parents")


def discover_models(root: Path) -> dict:
    """Return {model_folder: label}. Uses MODELS if given, otherwise every folder with results_* inside."""
    if MODELS:
        return dict(MODELS)
    found = sorted({p.parent.name for p in root.glob(f"*/{RESULTS_DIR_GLOB}") if p.is_dir()})
    return {name: name for name in found}


_TIMESTAMP_RE = re.compile(r"(\d{8}_\d{6})")

def latest_xlsx(results_dir: Path):
    """Newest .xlsx in a results_* folder: by the YYYYMMDD_HHMMSS stamp in the name, else by mtime."""
    files = [f for f in results_dir.glob("*.xlsx") if not f.name.startswith("~$")]
    if not files:
        return None
    def sort_key(f):
        m = _TIMESTAMP_RE.search(f.stem)
        return (m.group(1) if m else "", f.stat().st_mtime)
    return max(files, key=sort_key)


def source_from_dir(results_dir: Path, model_dir: str) -> str:
    """'results_lmc_spsa_resnet20_cifar10' with model 'resnet20_cifar10' -> 'lmc_spsa'."""
    name = results_dir.name
    if name.startswith("results_"):
        name = name[len("results_"):]
    suffix = f"_{model_dir}"
    if name.endswith(suffix):
        name = name[: -len(suffix)]
    return name


def load_results(root: Path, models: dict) -> pd.DataFrame:
    frames = []
    for model_dir in models:
        mdir = root / model_dir
        if not mdir.is_dir():
            print(f"[warn] model folder not found: {mdir}")
            continue
        for rdir in sorted(p for p in mdir.glob(RESULTS_DIR_GLOB) if p.is_dir()):
            f = latest_xlsx(rdir)
            if f is None:
                print(f"[warn] no .xlsx in {rdir.relative_to(root)}")
                continue
            df = pd.read_excel(f, sheet_name=SHEET_NAME)
            missing = {"method", ACC_COL, TIME_COL} - set(df.columns)
            if missing:
                print(f"[warn] {f.relative_to(root)} lacks columns {sorted(missing)}; skipped")
                continue
            if MEM_COL not in df.columns:
                df[MEM_COL] = np.nan          # memory is optional
            source = source_from_dir(rdir, model_dir)
            df["model"]  = model_dir
            df["source"] = source
            df["file"]   = str(f.relative_to(root))
            frames.append(df)
            print(f"{model_dir:26s} {source:12s} <- {f.relative_to(root)}  ({len(df)} rows)")
    if not frames:
        raise FileNotFoundError("No result spreadsheets were loaded")
    return pd.concat(frames, ignore_index=True)


PROJECT_ROOT = find_project_root()
models = discover_models(PROJECT_ROOT)
print(f"project root: {PROJECT_ROOT}\n")
raw = load_results(PROJECT_ROOT, models)

## Normalise labels, pick one source per method and compute the relative numbers

In [ ]:
_SUFFIX_RE = re.compile(r"\s*\([^()]*\)\s*$")   # trailing "(...)" e.g. "(T=1.0,10ep)"

def base_method(label) -> str:
    """Strip the trailing run details unless the full label is itself a known name."""
    label = str(label).strip()
    if label in DISPLAY_NAMES or label in EXCLUDE_METHODS:
        return label
    stripped = _SUFFIX_RE.sub("", label)
    return stripped or label


df = raw.copy()
df["method_base"] = df["method"].map(base_method)
df["method_name"] = df["method_base"].map(lambda b: DISPLAY_NAMES.get(b, b))
df = df[~df["method_base"].isin(EXCLUDE_METHODS)]

# A method that appears in several results_* folders (the baselines) is taken from PREFERRED_SOURCE
# when that folder reports it, otherwise from the first folder (alphabetically) that does.
priority = (df["source"] != PREFERRED_SOURCE).astype(int)
chosen = (df.assign(_prio=priority)
            .sort_values(["_prio", "source"])
            .groupby(["model", "method_name"])["source"].first()
            .rename("chosen_source"))
df = df.join(chosen, on=["model", "method_name"])
df = df[df["source"] == df["chosen_source"]].drop(columns="chosen_source")

# Mean over all rows of a method (every noise level and seed)
stats = (df.groupby(["model", "method_name"])
           .agg(acc=(ACC_COL, "mean"), time=(TIME_COL, "mean"), mem=(MEM_COL, "mean"), n=(ACC_COL, "count"))
           .reset_index())

def reference(name: str) -> pd.DataFrame:
    # the per-model averages of a reference method
    r = stats[stats["method_name"] == name].set_index("model")[["acc", "time", "mem"]]
    missing = [m for m in models if m not in r.index]
    if missing:
        raise ValueError(f"{name!r} not found for models {missing}")
    return r

acc_ref, time_ref = reference(ACC_REFERENCE), reference(TIME_REFERENCE)
# accuracy: gain over the unadapted model of the same model; time and memory: relative to backpropagation
stats["acc_gain_pct"] = 100.0 * (stats["acc"] / stats["model"].map(acc_ref["acc"]) - 1.0)
stats["rel_time"]     = stats["time"] / stats["model"].map(time_ref["time"])
stats["rel_mem"]      = stats["mem"]  / stats["model"].map(time_ref["mem"])
stats["speedup"]      = 1.0 / stats["rel_time"]

def ordered_methods(names) -> list:
    names = set(names)
    return [m for m in PLOT_ORDER if m in names] + sorted(names - set(PLOT_ORDER))

candidates = set(stats["method_name"]) - {ACC_REFERENCE, TIME_REFERENCE}
methods = ordered_methods(candidates if METHODS_TO_PLOT is None else candidates & set(METHODS_TO_PLOT))
points = stats[stats["method_name"].isin(methods)].copy()

# Average over the benchmarks: arithmetic mean of the accuracy gain, geometric mean of the speed-up. The time
# reference (backpropagation) is summarised too: it is drawn at 1x with its own accuracy gain.
summary = (stats[stats["method_name"].isin(methods + [TIME_REFERENCE])]
                 .groupby("method_name")
                 .agg(acc_gain_pct=("acc_gain_pct", "mean"),
                      speedup=("speedup", lambda v: float(np.exp(np.mean(np.log(v))))),
                      rel_mem=("rel_mem", "mean"),
                      n_models=("model", "nunique"))
                 .loc[methods + [TIME_REFERENCE]])
summary["rel_time"] = 1.0 / summary["speedup"]
partial = summary.index[summary["n_models"] < len(models)].tolist()
if partial:
    print(f"[warn] averaged over fewer benchmarks than the others: {partial}\n")

print("Sources:")
print(chosen.unstack("method_name").fillna("-").to_string(), "\n")
for model_dir, label in models.items():
    t = stats[stats["model"] == model_dir].set_index("method_name")
    t = t.loc[[ACC_REFERENCE, TIME_REFERENCE] + [m for m in methods if m in t.index]]
    print(f"== {label}")
    print(t[["acc", "time", "mem", "n", "acc_gain_pct", "speedup", "rel_mem"]].round(3).to_string(), "\n")

print(f"== Averaged over {len(models)} benchmarks (accuracy gain over {ACC_REFERENCE}: mean; "
      f"speed-up vs. {TIME_REFERENCE}: geometric mean)")
print(summary.round(3).to_string(), "\n")

# The highlighted method's gain per benchmark, as quoted in the text
HIGHLIGHT_GAINS = (stats[stats["method_name"] == HIGHLIGHT_METHOD].set_index("model")["acc_gain_pct"]
                   .reindex(list(models)))
print(f"Averaged across all noise levels, {HIGHLIGHT_METHOD} improves classification accuracy by "
      + ", ".join(f"{g:.1f}%" for g in HIGHLIGHT_GAINS.iloc[:-1]) + f", and {HIGHLIGHT_GAINS.iloc[-1]:.1f}% "
      + f"over the corresponding unadapted models ({', '.join(models.values())}), respectively.")

## Plot

In [ ]:
cycle = iter(plt.rcParams["axes.prop_cycle"].by_key()["color"] * 10)
colors = {m: COLORS.get(m) or next(cycle) for m in methods + [TIME_REFERENCE]}
GREEN, GREY, BLUE = "#2E7D32", "0.45", "#4C72B0"

# ---- fixed axis limits and equally spaced ticks ----
lo, hi = 0.2, 20.0

ticks = np.array([0.2, 0.3, 0.5, 0.7, 1, 2, 5, 10, 20])
positions = np.arange(len(ticks), dtype=float)

# Custom scale: equally spaced tick positions
# Interpolate logarithmically between the specified ticks
def interp_extrap(x, xp, fp):
    x = np.asarray(x, dtype=float)
    y = np.interp(x, xp, fp)

    left_slope = (fp[1] - fp[0]) / (xp[1] - xp[0])
    right_slope = (fp[-1] - fp[-2]) / (xp[-1] - xp[-2])

    y = np.where(x < xp[0], fp[0] + left_slope * (x - xp[0]), y)
    y = np.where(x > xp[-1], fp[-1] + right_slope * (x - xp[-1]), y)
    return y

def forward(x):
    x = np.asarray(x, dtype=float)
    x = np.maximum(x, np.finfo(float).tiny)
    return interp_extrap(np.log(x), np.log(ticks), positions)

def inverse(x):
    x = np.asarray(x, dtype=float)
    return np.exp(interp_extrap(x, positions, np.log(ticks)))

fig, ax = plt.subplots(figsize=FIGSIZE)

# ---- "faster than backpropagation" half, and the unadapted model (0 % gain) ----
ax.axvspan(lo, 1.0, facecolor=GREEN, alpha=0.055, zorder=0)
ax.axvline(1.0, color=GREY, lw=1.3, ls="--", alpha=0.9, zorder=1)
ax.axhline(0.0, color="black", lw=1.3, ls="--", alpha=0.75, zorder=1)

ax.text(
    0.97, REGION_LABEL_Y,
    "faster than\nbackpropagation",
    color=GREEN, fontsize=13, fontweight="bold",
    ha="right", va="bottom", linespacing=1.35,
    transform=ax.get_xaxis_transform()
)

ax.text(
    1.08, REGION_LABEL_Y,
    "slower than\nbackpropagation",
    color=GREY, fontsize=13, fontweight="bold",
    ha="left", va="bottom", linespacing=1.35,
    transform=ax.get_xaxis_transform()
)

ax.text(
    ZERO_LABEL_X, 5, "Unadapted model",
    color="black", fontsize=12, ha="right", va="bottom",
    transform=ax.get_yaxis_transform(), zorder=2
)

# ---- backpropagation: the 1x training-time reference, at its own accuracy gain ----
bp = summary.loc[TIME_REFERENCE]
# ax.scatter(
#     [1.0], [bp["acc_gain_pct"]],
#     s=330, marker="*",
#     facecolor=BLUE, edgecolor="black",
#     linewidth=1.1, zorder=5
# )

# ax.annotate(
#     "Backpropagation",
#     xy=(1.0, bp["acc_gain_pct"]),
#     xytext=(13, 14),
#     textcoords="offset points",
#     fontsize=13, color="#28456F",
#     ha="left", va="bottom",
#     linespacing=1.35, fontweight="bold"
# )

# ---- "better" arrows ----
ax.annotate(
    "", xy=(0.03, 0.90), xytext=(0.30, 0.90),
    xycoords="axes fraction",
    arrowprops=dict(
        arrowstyle="-|>", lw=2.6,
        color=GREEN, alpha=0.8
    )
)

ax.text(
    0.165, 0.94, "better",
    color=GREEN, fontsize=13, fontweight="bold",
    ha="center", va="center",
    transform=ax.transAxes
)

ax.annotate(
    "", xy=(0.05, 0.82), xytext=(0.05, 0.42),
    xycoords="axes fraction",
    arrowprops=dict(
        arrowstyle="-|>", lw=2.6,
        color=GREEN, alpha=0.8
    )
)

ax.text(
    0.078, 0.635, "better",
    color=GREEN, fontsize=14, fontweight="bold",
    ha="center", va="center", rotation=90,
    transform=ax.transAxes
)

# ---- one marker per method ----
for m in methods:
    r = summary.loc[m]
    is_ours = m == HIGHLIGHT_METHOD

    ax.scatter(
        r["rel_time"],
        r["acc_gain_pct"],
        s=MARKER_SIZE,
        marker="o",
        facecolor=colors[m],
        edgecolor="black",
        linewidth=2.2,
        alpha=0.9 if is_ours else 0.72,
        zorder=6 if is_ours else 4
    )

# ---- highlight: the per-benchmark gain of our method over the unadapted model ----
# if HIGHLIGHT_METHOD in methods:
#     r = summary.loc[HIGHLIGHT_METHOD]
#     lines = [f"{SHORT_NAMES.get(m, models[m])}: $\\bf{{{g:+.1f}\\%}}$" for m, g in HIGHLIGHT_GAINS.items()]
#     ax.annotate(
#         f"{HIGHLIGHT_METHOD} vs. unadapted model,\naveraged over all noise levels\n" + "\n".join(lines),
#         xy=(r["rel_time"], r["acc_gain_pct"]), xycoords="data",
#         xytext=HIGHLIGHT_BOX, textcoords="axes fraction",
#         fontsize=10, color="black", ha="left", va="bottom", linespacing=1.3, zorder=8,
#         bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=colors[HIGHLIGHT_METHOD], lw=1.8),
#         arrowprops=dict(arrowstyle="-|>", lw=1.8, color=colors[HIGHLIGHT_METHOD],
#                         shrinkA=0, shrinkB=11, connectionstyle="arc3,rad=0.15"),
#     )

# ---- axes ----
ax.set_xscale(
    "function",
    functions=(forward, inverse)
)

ax.set_xlim(lo, hi)
ax.set_ylim(*YLIM)

# Equally spaced ticks, with 1x centered
ax.set_xticks(ticks)
ax.set_xticklabels([f"{t:g}$\\times$" for t in ticks])

ax.xaxis.set_minor_locator(plt.NullLocator())
ax.yaxis.set_major_locator(MaxNLocator(nbins=6, steps=[1, 2, 5, 10]))

ax.set_xlabel(
    "Training time relative to backpropagation",
    fontsize=14
)
ax.set_ylabel(
    "Mean relative accuracy gain over\nunadapted model [%]",
    fontsize=14
)

ax.grid(True, which="major", alpha=0.22)

# ---- method legend ----
legend_order = (
    ([HIGHLIGHT_METHOD] if HIGHLIGHT_METHOD in methods else [])
    + [m for m in methods if m != HIGHLIGHT_METHOD]
)

handles = [
    Line2D(
        [], [],
        marker="o",
        ls="",
        markersize=11,
        markerfacecolor=colors[m],
        markeredgecolor="black",
        label=m + ("*" if m in partial else "")
    )
    for m in legend_order
]

ax.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=LEGEND_POS,
    ncol=LEGEND_NCOL,
    fontsize=11,
    frameon=False,
    columnspacing=1.0,
    handletextpad=0.3
)

fig.tight_layout()

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_FILE, dpi=300, bbox_inches="tight")

print(f"saved {OUTPUT_FILE.resolve()}")
plt.show()

## Per-benchmark view: speed-up and accuracy gain side by side

A "butterfly" chart, one row per benchmark. Method names sit in the middle; the **left wing** is the training speed-up
versus backpropagation (log scale, bars grow outward from the 1× line, so a bar pointing left means *faster* than full
retraining and a bar pointing inward means *slower*; backpropagation itself is the star on the 1× line); the **right
wing** is the accuracy gain over the unadapted model (the dashed 0 line), averaged over all noise levels. The gains differ by two orders of magnitude between the
benchmarks, so each row of the right wing has its own scale, with 0 at the same place in every row; the labels give
the exact values. Our method's values are the ones quoted in the text.

In [ ]:
# ---- settings for this plot ----
ROW_ORDER         = list(reversed(PLOT_ORDER))            # top-to-bottom order of the methods (Ours on top)
ROW_HEIGHT        = 1.35                                  # inches per benchmark row
ROW_GAP           = 0.40                                  # vertical gap between the benchmark rows (fraction of a row's axes height)
WING_WIDTH        = 10.0                                  # figure width in inches
BAR_H             = 0.68
GAIN_ZERO_AT      = None      # where 0 sits on every accuracy-gain axis (fraction of its width), the same in all rows;
                              # None -> the place that makes the longest bar of the most cramped row as long as possible
LABEL_GAP         = 0.015     # gap between a bar end and its value label (fraction of the axis width)
EDGE_MARGIN       = 0.01      # room between a value label and the end of its axis (fraction of the axis width)
OUTPUT_FILE_WINGS = Path("figures") / "per_benchmark_speedup_retention.png"

def fmt_speedup(sp: float) -> str:
    # one decimal above 1x, two below (so 0.047 reads 0.05 rather than 0.0)
    return f"{sp:.1f}" if sp >= 1 else f"{sp:.2f}"

row_methods = [m for m in ROW_ORDER if m in methods] + sorted(set(methods) - set(ROW_ORDER))
ypos = np.arange(len(row_methods))[::-1]                  # first method on top

# speed-up wing: log range shared by all rows, labelled at the decades only (bars carry the exact numbers)
sp_all = points["speedup"]
# symmetric in decades around 1x, so the 1x line is in the middle of the wing
n_dec = int(max(np.ceil(np.log10(max(sp_all.max() * 1.4, 1.0))), -np.floor(np.log10(min(sp_all.min() / 1.4, 1.0)))))
sp_hi, sp_lo = 10.0 ** n_dec, 10.0 ** -n_dec
sp_ticks = [10.0 ** k for k in range(-4, 5) if sp_lo <= 10.0 ** k <= sp_hi]

# the speed-up wings share one x axis; every accuracy-gain wing has its own
fig = plt.figure(figsize=(WING_WIDTH+1, ROW_HEIGHT * len(models)))
gs = fig.add_gridspec(len(models), 2, wspace=0.30, hspace=ROW_GAP)
axes = np.empty((len(models), 2), dtype=object)
for row in range(len(models)):
    axes[row, 0] = fig.add_subplot(gs[row, 0], sharex=axes[0, 0] if row else None)
    axes[row, 1] = fig.add_subplot(gs[row, 1])

def signed(v: float, fmt: str) -> str:
    # number with a typographic minus sign
    return format(v, fmt).replace("-", "\u2212")

pct = FuncFormatter(lambda v, _: signed(v, "g") + "%")

def gain_label(g: float) -> str:
    return signed(g, "+.1f") + "%"

# Each row of the accuracy-gain wing has its own scale, but 0 sits at the same place in all of them. That place and
# each row's scale are chosen so that every bar and its value label fit: the label widths are measured with the
# renderer, as fractions of the axis width.
renderer = fig.canvas.get_renderer()
gain_rows = []                                            # per row: [(gain, label width), ...]
for row, model_dir in enumerate(models):
    ax_a = axes[row, 1]
    ax_w = ax_a.get_window_extent(renderer).width
    sub = points[points["model"] == model_dir].set_index("method_name").reindex(row_methods)
    items = []
    for m in row_methods:
        g = sub.loc[m, "acc_gain_pct"]
        if np.isnan(g):
            continue
        t = ax_a.text(0.0, 0.0, gain_label(g), fontsize=10.5, fontweight="bold" if m == HIGHLIGHT_METHOD else "normal")
        items.append((float(g), t.get_window_extent(renderer).width / ax_w))
        t.remove()
    gain_rows.append(items)

def max_scale(items, zero_at: float) -> float:
    # the largest scale (axis fraction per percentage point) at which all bars and labels of a row fit; 0 if none does
    scale = np.inf
    for g, w in items:
        room = (1.0 - EDGE_MARGIN - zero_at - LABEL_GAP - w) if g >= 0 else (zero_at - EDGE_MARGIN - LABEL_GAP - w)
        if room <= 0:
            return 0.0
        if g != 0:
            scale = min(scale, room / abs(g))
    return scale

def longest_bar(items, zero_at: float) -> float:
    return max_scale(items, zero_at) * max(abs(g) for g, _ in items)

if GAIN_ZERO_AT is None:
    _grid = np.linspace(0.01, 0.99, 981)
    zero_at = float(_grid[np.argmax([min(longest_bar(it, z) for it in gain_rows) for z in _grid])])
else:
    zero_at = float(GAIN_ZERO_AT)
gain_scales = [max_scale(it, zero_at) for it in gain_rows]
if min(gain_scales) <= 0:
    raise ValueError(f"GAIN_ZERO_AT = {zero_at}: the value labels do not fit on both sides of 0")
print(f"0 of every accuracy-gain axis at {zero_at:.3f} of its width; longest bar per row: "
      + ", ".join(f"{longest_bar(it, zero_at):.2f}" for it in gain_rows) + " of the axis width")

for row, (model_dir, label) in enumerate(models.items()):
    ax_s, ax_a = axes[row]
    sub = points[points["model"] == model_dir].set_index("method_name").reindex(row_methods)
    g_width = 1.0 / gain_scales[row]                       # the row's axis width in percentage points
    g_lim = (-zero_at * g_width, (1.0 - zero_at) * g_width)

    for y, m in zip(ypos, row_methods):
        sp, gain = sub.loc[m, "speedup"], sub.loc[m, "acc_gain_pct"]
        if np.isnan(sp) or np.isnan(gain):
            continue
        is_ours = m == HIGHLIGHT_METHOD
        kw = dict(color=colors[m], edgecolor="black", linewidth=0.8,
                  alpha=1.0 if is_ours else 0.8, zorder=3, height=BAR_H)
        txt = dict(va="center", fontsize=10.5, fontweight="bold" if is_ours else "normal",
                   color=colors[m] if is_ours else "black")
        # left wing: bar between 1x and the speed-up (log axis is inverted, so >1 grows outward to the left)
        ax_s.barh(y, width=abs(sp - 1.0), left=min(sp, 1.0), **kw)
        if sp >= 1.0:
            ax_s.text(sp * 1.1, y, f"{fmt_speedup(sp)}$\\times$", ha="right", **txt)
        else:
            ax_s.text(sp / 1.1, y, f"{fmt_speedup(sp)}$\\times$", ha="left", **txt)
        # right wing: accuracy gain over the unadapted model (negative = worse than not adapting)
        ax_a.barh(y, width=gain, left=0.0, **kw)
        off = LABEL_GAP * g_width
        ax_a.text(gain + off if gain >= 0 else gain - off, y, gain_label(gain),
                  ha="left" if gain >= 0 else "right", **txt)

    # left wing cosmetics
    ax_s.set_xscale("log")
    ax_s.set_xlim(sp_hi, sp_lo)                            # inverted: larger speed-up further left (outward)
    ax_s.axvspan(1.0, sp_hi, facecolor=GREEN, alpha=0.055, zorder=0)
    ax_s.axvline(1.0, color=GREY, lw=1.2, ls="--", zorder=1)
    ax_s.set_xticks(sp_ticks)
    ax_s.set_xticklabels([f"{t:g}$\\times$" for t in sp_ticks])
    ax_s.xaxis.set_minor_locator(plt.NullLocator())
    ax_s.tick_params(axis="x", labelbottom=row == len(models) - 1)
    ax_s.set_yticks(ypos)
    ax_s.tick_params(axis="y", labelleft=False, length=0)
    ax_s.set_ylim(-0.6, len(row_methods) - 0.4)
    ax_s.grid(True, axis="x", alpha=0.22, zorder=0)
    ax_s.set_title(label, loc="left", fontsize=12, fontweight="bold", pad=4)
    for side in ("top", "right"):
        ax_s.spines[side].set_visible(False)

    # right wing cosmetics (method names in the middle)
    ax_a.set_xlim(*g_lim)
    ax_a.xaxis.set_major_locator(MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10]))
    ax_a.xaxis.set_major_formatter(pct)
    # a gain below -100 % is impossible: the axis may extend that far only to keep 0 aligned or to fit a label
    ax_a.set_xticks([t for t in ax_a.get_xticks() if max(-100.0, g_lim[0]) <= t <= g_lim[1]])
    ax_a.tick_params(axis="x", labelsize=11)
    ax_a.axvline(0.0, color="black", lw=1.2, ls="--", alpha=0.75, zorder=1)
    ax_a.set_yticks(ypos)
    ax_a.set_yticklabels(row_methods, fontsize=12)
    for tl, m in zip(ax_a.get_yticklabels(), row_methods):
        if m == HIGHLIGHT_METHOD:
            tl.set_fontweight("bold"); tl.set_color(colors[HIGHLIGHT_METHOD])
    ax_a.tick_params(axis="y", length=0)
    ax_a.set_ylim(-0.6, len(row_methods) - 0.4)
    ax_a.grid(True, axis="x", alpha=0.22, zorder=0)
    for side in ("top", "right", "left"):
        ax_a.spines[side].set_visible(False)


# backpropagation's own speed, 1x: a star on the dashed 1x line above the first row, as in the trade-off plot
# axes[0, 0].plot([1.0], [1.09], marker="*", ms=17, mfc=BLUE, mec="black", mew=1.0, ls="",
#                 transform=axes[0, 0].get_xaxis_transform(), clip_on=False, zorder=6)
# axes[0, 0].annotate(f"{TIME_REFERENCE} (1$\\times$)", xy=(1.0, 1.09), xycoords=axes[0, 0].get_xaxis_transform(),
#                     xytext=(9, 0), textcoords="offset points", ha="left", va="center",
#                     fontsize=12, fontweight="bold", color="#28456F", annotation_clip=False)

axes[0, 1].text(15.0, 1.1, ACC_REFERENCE, transform=axes[0, 1].get_xaxis_transform(), ha="center", va="bottom",
                fontsize=14, color=GREY)

axes[-1, 0].set_xlabel("Speed-up vs. backpropagation", fontsize=14)
axes[-1, 1].set_xlabel("Relative accuracy gain over unadapted model [%]", fontsize=14)

OUTPUT_FILE_WINGS.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_FILE_WINGS, dpi=300, bbox_inches="tight")
print(f"saved {OUTPUT_FILE_WINGS.resolve()}")
plt.show()